# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



In [1]:
import zipfile

zip_path = '/content/2026-ssafy-15-2-ai (2).zip'
extract_path = '/content/ai_challenge'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print('압축 해제 완료')

압축 해제 완료


# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [2]:
#!pip -q install git+https://github.com/huggingface/transformers accelerate
#!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
#!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [4]:
#!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

Looking in indexes: https://download.pytorch.org/whl/nightly/cu128


In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [2]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 압축 해제
!unzip "/content/drive/My Drive/260401_15_2_ai_Data.zip" -d "/content/"

Archive:  /content/drive/My Drive/260401_15_2_ai_Data.zip
replace /content/dev.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

# 라이브러리, 데이터, 설정

In [17]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
import gc
from typing import Dict, List, Any
from transformers import (
    AutoModelForImageTextToText, # Qwen-VL 계열 정석 클래스
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ✅ 모델 ID Qwen3-VL-8B-Instruct로 최종 확정!
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
IMAGE_SIZE = 512
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")
print(f"✅ 전체 학습 데이터 개수: {len(train_df)}개")

train_df = train_df.sample(n=500, random_state=SEED).reset_index(drop=True)

Device: cuda
✅ 전체 학습 데이터 개수: 5073개


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [6]:
# ✅ 이미 설치 완료되었으므로 주석 처리 (필요할 때만 다시 실행)
#!pip install hf_xet


In [18]:
# ✅ 2. 양자화 설정 (RTX 5060 Ti는 bfloat16을 네이티브로 지원하므로 변경 권장)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # float16보다 학습 안정성이 훨씬 높습니다.
)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,   # 최소 픽셀 수 제한
    max_pixels=1024 * 28 * 28,  # 최대 픽셀 수 제한 (VRAM 터지면 768로 낮추세요) -> 1024에서 512로 타협
    trust_remote_code=True,
)

# 사전학습 모델
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    r=16, # 16에서 8로 타협 -> 다시 16
    lora_alpha=32, # 32에서 16으로 타협
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)



# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [8]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [19]:
# 커스텀 데이터셋
import random
from PIL import Image
from torch.utils.data import Dataset
from dataclasses import dataclass
from typing import Any

# ✅ 데이터 증강(train_transform) 완전 제거됨

class VQAMCDataset(Dataset):
    def __init__(self, df, processor, is_train=True, has_label=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.is_train = is_train     # 학습 모드 (셔플링용)
        self.has_label = has_label   # 라벨 모드 (정답 텍스트 추가)

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        # 🔥 원본 이미지 훼손 방지를 위해 증강 로직 제거

        # 원본 선지와 정답
        options = {"a": row["a"], "b": row["b"], "c": row["c"], "d": row["d"]}
        ans_key = str(row["answer"]).strip().lower()
        ans_text = options[ans_key] # 진짜 정답 내용 (예: "공원")

        # 🔥 마법의 셔플링: 학습 시에만 선지 순서를 무작위로 섞음!
        keys = ["a", "b", "c", "d"]
        if self.is_train: # 학습 시에만 섞음
            random.shuffle(keys)

        # 섞인 순서대로 새로운 a,b,c,d 배정
        new_a, new_b, new_c, new_d = options[keys[0]], options[keys[1]], options[keys[2]], options[keys[3]]

        # 바뀐 선지들 중, 진짜 정답 내용("공원")이 어디로 갔는지 찾아서 새 정답 지정
        new_ans_key = ["a", "b", "c", "d"][keys.index(ans_key)]

        # 섞인 상태로 프롬프트 생성
        user_text = build_mc_prompt(str(row["question"]), new_a, new_b, new_c, new_d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.has_label: # 학습과 검증 모두 Loss 계산을 위해 정답 추가
            gold = new_ans_key
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        prompt_lengths = [] # 🔥 프롬프트 길이 추적용 리스트 추가

        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            # 전체 텍스트 렌더링
            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

            if self.train:
                # 🔥 정답을 제외한 '프롬프트' 부분만의 길이를 미리 계산
                prompt_messages = messages[:-1] # assistant의 정답 딕셔너리 제외
                prompt_text = self.processor.apply_chat_template(
                    prompt_messages,
                    tokenize=False,
                    add_generation_prompt=True
                )

                # 프롬프트만 따로 토크나이징하여 길이 파악
                prompt_enc = self.processor(
                    text=[prompt_text],
                    images=[img],
                    return_tensors="pt"
                )
                prompt_lengths.append(prompt_enc["input_ids"].shape[1])

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()
            # 패딩 부분 -100 처리
            labels[enc["attention_mask"] == 0] = -100

            # 🔥 핵심 방어막: User 프롬프트 부분은 채점 제외 (-100)
            for i, p_len in enumerate(prompt_lengths):
                labels[i, :p_len] = -100

            enc["labels"] = labels

        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [20]:
# 🚨 데이터 분리 전에 무조건 전체 데이터를 한 번 골고루 섞어줍니다!
shuffled_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# 검증용 데이터 분리 (섞인 데이터에서 9:1로 나누기)
split = int(len(shuffled_df) * 0.9)
train_subset, valid_subset = shuffled_df.iloc[:split], shuffled_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, is_train=True, has_label=True)
valid_ds = VQAMCDataset(valid_subset, processor, is_train=False, has_label=True)
test_ds = VQAMCDataset(test_df, processor, is_train=False, has_label=False) # 추론용이 별도로 있다면

# 데이터로더
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [21]:
import os
from tqdm.auto import tqdm
from transformers import get_cosine_schedule_with_warmup
import math
import gc # 🔥 VRAM 관리를 위한 라이브러리 추가

model = model.to(device)
GRAD_ACCUM = 4
EPOCHS = 3
SAVE_EVERY_N_STEPS = 100  # 🚨 몇 스텝마다 중간 저장할지 설정

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.1), # 초반 10%는 워밍업
    num_training_steps=num_training_steps
)

# 학습 루프 진입 전 초기화
optimizer.zero_grad(set_to_none=True)

# 💾 저장 경로 설정
SAVE_DIR = "./qwen3_8b_lora"              # 베스트 모델 저장 경로 (최종 추론용)
CHECKPOINT_DIR = "./latest_checkpoint"      # 중간 체크포인트 저장 경로
STATE_FILE = os.path.join(CHECKPOINT_DIR, "training_state.pt")

# 베스트 Loss 추적용 변수
# 기본 상태 변수 초기화
start_epoch, resume_step, global_step = 0, 0, 0
best_val_loss = float('inf')

# 🔄 체크포인트 불러오기 로직 추가
if os.path.exists(STATE_FILE):
    print(f"🔄 이전 학습 상태를 불러옵니다: {STATE_FILE}")
    checkpoint = torch.load(STATE_FILE, map_location=device)

    start_epoch = checkpoint['epoch']
    resume_step = checkpoint['step']
    global_step = checkpoint['global_step']
    best_val_loss = checkpoint['best_val_loss']

    # 모델 어댑터 및 옵티마이저/스케줄러 상태 복구
    model.load_adapter(CHECKPOINT_DIR, adapter_name="default")
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    print(f"✅ Epoch {start_epoch+1}, Step {resume_step} 부터 이어서 시작합니다!")
else:
    print("🆕 저장된 체크포인트가 없습니다. 처음부터 학습을 시작합니다.")

# 학습 루프
for epoch in range(start_epoch, EPOCHS):
    model.train()
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        # ⏩ 이어서 시작할 때, 이미 진행했던 스텝은 패스
        if epoch == start_epoch and step <= resume_step:
            continue

        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        # 스케일러 없이 깔끔하게 바로 backward!
        loss.backward()

        # 매 스텝마다 현재 원본 Loss를 진행바에 즉각 반영하여 부드럽게 표시!
        progress_bar.set_postfix({"loss": f"{outputs.loss.item():.4f}"})

        # step이 마지막 배치에 도달했을 때도 업데이트 강제 실행!
        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

        # 💾 일정 스텝마다 중간 체크포인트 저장 추가!
        if step % SAVE_EVERY_N_STEPS == 0:
            print(f"\n💾 [Step {step}] 중간 체크포인트를 저장합니다...")
            model.save_pretrained(CHECKPOINT_DIR) # LoRA 가중치 저장
            torch.save({
                'epoch': epoch,
                'step': step,
                'global_step': global_step,
                'best_val_loss': best_val_loss,
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
            }, STATE_FILE) # 학습 상태 저장

    # 에포크가 끝나면 다음 에포크를 위해 resume_step 초기화
    resume_step = 0

    # 검증 (Validation) 루프
    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k: v.to(device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1

    avg_val_loss = val_loss / val_steps
    print(f"[Epoch {epoch+1}] valid loss {avg_val_loss:.4f}")

    # ✅ 베스트 모델 저장 로직 (Loss가 줄어들 때만 덮어쓰기 저장)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"🔥 Best 모델 갱신! (Loss: {best_val_loss:.4f}) 모델을 저장합니다: {SAVE_DIR}")
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)

    # 🧹 VRAM 관리: 매 에포크가 끝날 때마다 찌꺼기 메모리 강제 정리
    gc.collect()
    torch.cuda.empty_cache()

print("🎉 모든 학습이 완료되었습니다!")

print("💾 베스트 모델 가중치를 불러와 적용합니다...")
# adapter_name="default" 파라미터를 추가하여 어떤 어댑터에 덮어씌울지 명시합니다.
model.load_adapter(SAVE_DIR, adapter_name="default")
model.eval() # 추론 모드 전환

print("✅ 성공적으로 베스트 모델이 준비되었습니다! 이제 추론을 진행하세요.")

🆕 저장된 체크포인트가 없습니다. 처음부터 학습을 시작합니다.


Epoch 1/3 [train]:   0%|          | 0/450 [00:00<?, ?batch/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)



💾 [Step 100] 중간 체크포인트를 저장합니다...

💾 [Step 200] 중간 체크포인트를 저장합니다...

💾 [Step 300] 중간 체크포인트를 저장합니다...

💾 [Step 400] 중간 체크포인트를 저장합니다...


Epoch 1 [valid]:   0%|          | 0/50 [00:00<?, ?batch/s]

...

[Epoch 2] valid loss 0.0710
🔥 Best 모델 갱신! (Loss: 0.0710) 모델을 저장합니다: ./qwen3_8b_lora


Epoch 3/3 [train]:   0%|          | 0/450 [00:00<?, ?batch/s]


💾 [Step 100] 중간 체크포인트를 저장합니다...

💾 [Step 200] 중간 체크포인트를 저장합니다...

💾 [Step 300] 중간 체크포인트를 저장합니다...

💾 [Step 400] 중간 체크포인트를 저장합니다...


Epoch 3 [valid]:   0%|          | 0/50 [00:00<?, ?batch/s]

[Epoch 3] valid loss 0.0779
🎉 모든 학습이 완료되었습니다!
💾 베스트 모델 가중치를 불러와 적용합니다...
✅ 성공적으로 베스트 모델이 준비되었습니다! 이제 추론을 진행하세요.


In [22]:
import os
import re
import pandas as pd
import torch
from tqdm.auto import tqdm
from PIL import Image

SAVE_DIR = "./qwen3_8b_lora"
CHECKPOINT_DIR = "./latest_checkpoint"

# 1. 🚨 1에폭 학습 가중치 안전하게 불러오기 로직
# 1에폭 검증(valid)이 끝나서 Best 모델이 저장되었는지, 아니면 중간 체크포인트만 있는지 확인합니다.
if os.path.exists(SAVE_DIR):
    print(f"🔥 1에폭 완료 후 저장된 Best 모델({SAVE_DIR})을 불러옵니다.")
    model.load_adapter(SAVE_DIR, adapter_name="default")
elif os.path.exists(CHECKPOINT_DIR):
    print(f"⚠️ Best 모델이 없어 가장 마지막에 저장된 중간 체크포인트({CHECKPOINT_DIR})를 불러옵니다.")
    model.load_adapter(CHECKPOINT_DIR, adapter_name="default")
else:
    print("⚠️ 저장된 폴더가 없습니다! 현재 VRAM에 남아있는 학습 상태 그대로 추론을 진행합니다.")

model.eval() # 추론 모드 전환
print("✅ 성공적으로 모델이 준비되었습니다! 추론을 시작합니다.")

🔥 1에폭 완료 후 저장된 Best 모델(./qwen3_8b_lora)을 불러옵니다.
✅ 성공적으로 모델이 준비되었습니다! 추론을 시작합니다.


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [23]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    # 1. 정규표현식으로 a, b, c, d 중 하나만 독립적으로 존재하는지 탐색
    # 예: "(a)", "a.", "정답은 a입니다" 등 모두 커버
    match = re.search(r'\b([abcd])\b', text)
    if match:
        return match.group(1)

    # 2. 정규식으로 못 찾았을 경우 기존의 줄바꿈/토큰 탐색 로직 (Fallback)
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok

    return "a" # 못 찾으면 기본값

# 추론을 위해 모든 레이어 활성화
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        # max_new_tokens를 넉넉하게 8로 줍니다.
        out_ids = model.generate(**inputs, max_new_tokens=8, do_sample=False,
                         eos_token_id=processor.tokenizer.eos_token_id,
                         pad_token_id=processor.tokenizer.pad_token_id)

    # 🚨 누락된 핵심 코드 추가 (프롬프트 빼고 진짜 대답만 자르기)
    input_len = inputs.input_ids.shape[1]
    generated_ids = out_ids[:, input_len:]

    # 잘라낸 대답만 텍스트로 변환
    output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    preds.append(extract_choice(output_text))

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("./submission.csv", index=False)
print("submission.csv")

Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


submission.csv


In [ ]:
# 모델 응답 예시
print(output_text)